# Mã nguồn thu thập dữ liệu từ Foody

In [22]:
import re
import pandas as pd
import numpy as np
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.action_chains import ActionChains
import random
import time
from foody_opt_v14 import *

### Các hàm phụ trợ

In [23]:
# Write file txt function
def write_file_txt(filename, list_data):
    try:

        with open(filename, "w") as f:
            for item in list_data:
                f.write(item)
    except:
        print('Lỗi không ghi được file')

# Read file txt function
def read_file_txt(filename):
    with open(filename, "r") as f:
        lines = f.readlines()
    return [line.rstrip("\n") for line in lines]

# Simulate human actions
def human_action_simulation(driver):
    """
    Giả lập các thao tác: cuộn trang, di chuyển chuột và nghỉ ngẫu nhiên.
    """
    # 1. Giả lập cuộn trang (Scroll)
    # Thay vì cuộn một lèo, ta chia nhỏ thành nhiều đợt cuộn ngắn dài khác nhau
    total_height = driver.execute_script("return document.body.scrollHeight")
    current_pos = driver.execute_script("return window.pageYOffset")
    
    scroll_steps = random.randint(3, 6)
    for _ in range(scroll_steps):
        scroll_distance = random.randint(200, 600)
        driver.execute_script(f"window.scrollBy(0, {scroll_distance});")
        # Nghỉ ngắn sau mỗi lần cuộn để giả vờ như đang đọc nội dung
        time.sleep(np.random.uniform(0.5, 1))

    # 2. Giả lập di chuyển chuột (Mouse Movement)
    # Di chuyển chuột đến một tọa độ ngẫu nhiên trên màn hình
    try:
        actions = ActionChains(driver)
        # Lấy kích thước cửa sổ trình duyệt
        width = driver.get_window_size()['width']
        height = driver.get_window_size()['height']
        # Di chuyển đến một vài điểm ngẫu nhiên
        for _ in range(random.randint(2, 4)):
            x_offset = random.randint(0, width - 1)
            y_offset = random.randint(0, height - 1)
            time.sleep(np.random.uniform(0.5, 1))
            try: 
                actions.move_by_offset(x_offset, y_offset).perform() # Nếu offset là hợp lệ
            except:
                pass # Không di chuyển chuột khi OutOfBound
            # Reset vị trí chuột về (0,0) để tránh lỗi di chuyển ra ngoài màn hình ở vòng lặp sau
            actions.move_to_element_with_offset(driver.find_element("tag name", "body"), 0, 0).perform()
    except Exception as e:
        print(f"Lỗi khi giả lập chuột: {e}")
    time.sleep(np.random.uniform(0.5, 1))    

chrome_options = Options()

def login(driver):
    driver.get("https://id.foody.vn/account/login")
    wait = WebDriverWait(driver, 5)

    # wait.until(EC.presence_of_element_located((By.NAME, "Minh Quang")))
    driver.find_element(By.NAME, "Email").send_keys("nguyenquangminh772006@gmail.com")
    driver.find_element(By.NAME, "Password").send_keys("24521075")
    driver.find_element(By.XPATH, '//*[@id="bt_submit"]').click()

    # Chờ redirect sau khi login
    time.sleep(np.random.uniform(1, 2))
    print("URL sau login:", driver.current_url)
    driver.get("http://foody.vn")
    time.sleep(np.random.uniform(1, 2))

In [ ]:
def scroll_and_collect_links(driver, max_links=300):
    i = 1
    list_links = []
    processed_links = set() # Dùng set để lọc trùng ngay lập tức
    count_item = None
    try:
        count_item = driver.find_element(By.CLASS_NAME, 'result-status-count').text
    except:
        count_item = 999  # Giá trị mặc định nếu không lấy được
    print(f'Total expected: {(count_item)}')

    previous_num = -1
    while len(list_links) <= max_links:
        
        # 1. Cuộn trang để load dữ liệu (nếu là dạng infinite scroll)
        scroll_steps = random.randint(2, 4)
        for _ in range(scroll_steps):
            driver.execute_script(f"window.scrollBy(0, {random.randint(300, 600)});")
            time.sleep(random.uniform(0.8, 1.5))

        # 2. Lấy danh sách nhà hàng ở trang HIỆN TẠI
        div_list_restaurant = driver.find_elements(By.CLASS_NAME, "resname")
        
        for div_restaurant in div_list_restaurant:
            try:
                link = div_restaurant.find_element(By.XPATH, ".//a[@target='_blank']").get_attribute('href')
                
                if link and link not in processed_links:
                    list_links.append(link + '\n')
                    processed_links.add(link)
            except Exception:
                continue
        if previous_num == len(list_links):
            print('Hết link!')
            break
        previous_num = len(list_links)
        
        # --- GHI FILE SAU MỖI VÒNG LẶP TẠI ĐÂY ---
        if list_links:
            write_file_txt('../../data/data_raw/txt/link_food_hcm.txt', list_links)
            print(f'Đã backup tạm thời {len(list_links)} links vào file.')
        try:        
            # Tìm element có id scrollLoadingPage
            next_button = driver.find_element(By.ID, 'scrollLoadingPage')
            
            # Kiểm tra xem nó là link hay button
            tag_name = next_button.tag_name
            
            if tag_name == 'a':
                # Nếu là thẻ <a>, lấy href
                next_link = next_button.get_attribute('href')
            else:
                driver.execute_script("arguments[0].click();", next_button)
                time.sleep(1.5)
                i += 1
                continue  # Quay lại vòng lặp, không cần get link mới
        
        except Exception as e:
            print(f"Lỗi hoặc hết trang: {e}")
            break

    # 5. Lưu file
    if list_links:
        write_file_txt('../../data/data_raw/txt/link_food_hcm.txt', list_links)
        print(f'Done! Saved {len(list_links)} links.')
    return list_links

In [25]:
def get_links(area_url, max_links=300):
    chrome_options = Options()
    user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"

    chrome_options.add_argument(f'user-agent={user_agent}')
    chrome_options.add_argument("--disable-blink-features=AutomationControlled") # Ẩn cờ "bot" của Selenium
    chrome_options.exclude_switches = ["enable-automation"]
    chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
    chrome_options.add_experimental_option('useAutomationExtension', False)
    chrome_options.use_automation_extension = False
    driver = webdriver.Chrome(options = chrome_options)
    driver.maximize_window()
    login(driver)
    driver.get(area_url)
    time.sleep(2)
    # Ghi links crawled vào file txt
    scroll_and_collect_links(driver, max_links)    
    driver.quit()

### Vui lòng điền link địa điểm và số quán tối đa (max_links) cần cào tại đây

In [26]:
get_links("https://www.foody.vn/an-giang/food/dia-diem?q=m%C3%AC+cay&ss=header_search_form", max_links=32)

URL sau login: https://id.foody.vn/tai-khoan
Total expected: 75 kết quả
"mì cay"
 ở ăn uống
Đã backup tạm thời 24 links vào file.
Đã backup tạm thời 34 links vào file.
Done! Saved 34 links.


### Phần cào dữ liệu
* Vui lòng chọn số hiệu file đầu ra ở tham số **out_name** trong hàm **run_parallel_scraper_with_checkpoint**

In [27]:
# ═══════════════════════════════════════════════════════════════════════════════
# MAIN
# ═══════════════════════════════════════════════════════════════════════════════
if __name__ == "__main__":
    LINK_FILE = os.path.join(TXT_DIR, "link_food_hcm.txt")

    # Bước 0: Đảm bảo đã đăng nhập Foody
    if not ensure_foody_cookies():
        log.error("Đăng nhập Foody thất bại. Dừng chương trình.")
        exit(1)

    if os.path.exists(LINK_FILE):
        with open(LINK_FILE, encoding="utf-8") as f:
            links = [l.strip() for l in f if l.strip()]
        log.info(f"Đọc {len(links)} links từ file")
    

    # Bước 2: Cào song song
    df_quan, df_bl = run_parallel_scraper_with_checkpoint(links, max_workers=MAX_WORKERS, out_name="1000")

2026-06-06 23:54:30,520 [INFO] Cookie Foody đã có sẵn: ../../data/data_raw/foody_csv\foody_cookies.json
2026-06-06 23:54:30,554 [INFO] Đọc 34 links từ file
2026-06-06 23:54:30,556 [INFO] Cào 34 link mới với 6 workers...
2026-06-06 23:54:30,557 [INFO] Khởi động 6 Chrome drivers (có session Foody)...
2026-06-06 23:54:36,615 [INFO] ✓ Đã inject cookies Foody vào driver.
2026-06-06 23:54:42,420 [INFO] ✓ Đã inject cookies Foody vào driver.
2026-06-06 23:54:49,381 [INFO] ✓ Đã inject cookies Foody vào driver.
2026-06-06 23:55:01,222 [INFO] ✓ Đã inject cookies Foody vào driver.
2026-06-06 23:55:11,561 [INFO] ✓ Đã inject cookies Foody vào driver.
2026-06-06 23:55:19,391 [INFO] ✓ Đã inject cookies Foody vào driver.
2026-06-06 23:55:19,393 [INFO] Driver pool sẵn sàng (6 drivers).
2026-06-06 23:55:29,452 [INFO] ✓ [3] Mì Cay Hải Sản SuBi | 0 reviews
2026-06-06 23:55:29,706 [INFO] ✓ [5] Trà Sữa Mì Cay Ăn Vặt - Ti Mon | 0 reviews
2026-06-06 23:55:30,208 [INFO] ✓ [2] Mỳ Cay Kibou Coffee & Tea | 0 revie